Backpropagation by hand, because an in-depth understanding of how backpropagation works under the hood is important for debugging neural networks down the road.  

This notebook begins with the most stepwise version of backpropagation and gradually reduces the steps by compressing into neater mathematical expressions.

In [1]:
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn.functional as F

In [2]:
# read in words 
words = open("data/names.txt", "r").read().splitlines()

# Construct mappings between characters and indicies
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)} # string-to-index
stoi["."] = 0
itos = {i:s for s, i in stoi.items()} # index-to-string
vocab_size = len(itos)

# Build the dataset, with splitting of input sets
blocksize = 3

# Set the context length 
def build_dataset(words, blocksize=blocksize):
    blocksize = blocksize
    X, Y = [], []

    for w in words:
        context = [0] * blocksize
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print("".join(itos[i] for i in context), "--->", itos[ix])
            context = context[1:] + [ix] 

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

# shuffle up the input data, and split up the data
import random
random.seed(67)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1], blocksize=blocksize)
Xdev, Ydev = build_dataset(words[n1:n2], blocksize=blocksize)
Xtest, Ytest = build_dataset(words[n2:], blocksize=blocksize)

In [3]:
# Utility function for comparing gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff {maxdiff}")

In [4]:
n_embed = 10
n_hidden = 200
batch_size = 32

# Initialize the NN params
g = torch.Generator().manual_seed(67)
C = torch.randn((vocab_size, n_embed), generator=g) # 10 D n_embed
# Layer 1
W1 = torch.randn((n_embed * blocksize, n_hidden), generator=g) * (5/3) / (n_embed * blocksize)**0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# Parameters, etc. related to batch normalization
bngain = torch.ones((1, n_hidden))*0.1 + 1.0
bnbias = torch.zeros((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True
print(sum(p.nelement() for p in parameters))

12297


In [5]:
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [6]:
# Forward pass
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# Linear layer 1
hprebn = embcat @ W1 + b1

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Nonlinear activation
h = torch.tanh(hpreact)

# Linear layer 2
logits = h @ W2 + b2

# Compute cross entropy loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# Back pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnvar,
          bnvar, bnvar_inv, bnraw, bnmeani, bndiff, bndiff2,
          bnvar_inv, hprebn, embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.6644, grad_fn=<NegBackward0>)

In [7]:
# probs = counts * counts_sum_inv
print(probs.shape)
print(counts.shape)
print(counts_sum_inv.shape)

torch.Size([32, 27])
torch.Size([32, 27])
torch.Size([32, 1])


My attempt at implenting stepwise back propagation, by going through each step in the forward pass. I do these without the tutorial first here. Below is my follow along from the tutorial.

In [8]:
# My own derivative of the loss wrt all elements of logprobs
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1/n

dprobs = dlogprobs / probs 

# wrong! In this step, I didn't account for broadcasting
dcounts_sum_inv = counts * dprobs 

dcounts_sum = -(counts_sum)**-2 * dcounts_sum_inv

dcounts = counts_sum_inv * dprobs + torch.ones_like(counts) * dcounts_sum

dnorm_logits = norm_logits.exp() * dcounts

# wrong! In this step, I didn't account for broadcasting
dlogit_maxes = -dnorm_logits 

# wrong! An incorrect implementation of plucking out the logit maxes
# a better strategy is to use one_hot; then all the unneeded derivatives are just zero
dlogits = dnorm_logits + dlogit_maxes[logits.max(1).indices]


dh = dlogits @ W2.T

dW2 = h.T @ dlogits

# wrong! In this step, I didn't account for broadcasting
db2 = dlogits


x = torch.tanh(hpreact)

dhpreact = dh * (1 - x**2)

# wrong! In this step, I didn't account for broadcasting
dbngain = dhpreact * bnraw

dbnraw = dhpreact * bngain

# wrong! In this step, I didn't account for broadcasting
dbnbias = dhpreact 

# wrong! In this step, I didn't account for broadcasting
dbnvar_inv = dbnraw * bndiff

dbnvar = dbnvar_inv * -0.5 * (bnvar + 1e-5)**-1.5

dbndiff2 = dbnvar * 1/(n-1) * torch.ones_like(bndiff2)

dbndiff = dbnraw * bnvar_inv + dbndiff2 * 2 * bndiff

dhprebn = dbndiff 

dbnmeani = -dbndiff  

dhprebn = dbnmeani * 1/n * torch.ones_like(hprebn)

# (m x n)   (m x c)(c x n)
dembcat = dhprebn @ W1.T

dW1 = embcat.T @ dhprebn 

# wrong! In this step, I didn't account for broadcasting
db1 = dhprebn

demb = dembcat.view(emb.shape)

# wrong! I just had no idea what to do here :)
dC = torch.zeros_like(C)
dC[Xb] += demb

When I first went through this, I didn't pay attention to broadcasting. Then I realized the shapes of the tensors always need to be considered. I went back through each step on my own, trying to get it to work before seeing the reveal. Stepwise using the helper function `cmp()` makes this process far easier!

In [9]:
# Tutorial derivative of the loss wrt all elements of logprobs
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1/n
cmp("logprobs", dlogprobs, logprobs)

dprobs = dlogprobs / probs 
cmp("dprobs", dprobs, probs)

# In this step, I didn't account for broadcasting
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp("dcounts_sum_inv", dcounts_sum_inv, counts_sum_inv)

dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
cmp("dcounts_sum", dcounts_sum, counts_sum)

dcounts = counts_sum_inv * dprobs + torch.ones_like(counts) * dcounts_sum
cmp("dcounts", dcounts, counts)

dnorm_logits = norm_logits.exp() * dcounts
cmp("dnorm_logits", dnorm_logits, norm_logits)

# In this step, I didn't account for broadcasting
dlogit_maxes = - dnorm_logits.sum(1, keepdim=True)
cmp("dlogit_maxes", dlogit_maxes, logit_maxes)

# In this step, I realized I needed logits.max(1).indices, 
# but I should have used one hot to get the structure I need
dlogits = dnorm_logits.clone() + F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
cmp("dlogits", dlogits, logits)


# logits = h @ W2 + b2
dh = dlogits @ W2.T
cmp("dh", dh, h)
dW2 = h.T @ dlogits
cmp("dW2", dW2, W2)
db2 = dlogits.sum(0, keepdim=True)
cmp("db2", db2, b2)

# # h = torch.tanh(hpreact)
x = torch.tanh(hpreact)
dhpreact = dh * (1 - x**2)
cmp("dhpreact", dhpreact, hpreact)
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
cmp("dbngain", dbngain, bngain)
dbnraw = dhpreact * bngain
cmp("dbnraw", dbnraw, bnraw)
dbnbias = dhpreact.sum(0, keepdim=True)
cmp("dbnbias", dbnbias, bnbias)
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp("dbnvar_inv", dbnvar_inv, bnvar_inv)
dbnvar = dbnvar_inv * -0.5 * (bnvar + 1e-5)**-1.5
cmp("dbnvar", dbnvar, bnvar)
dbndiff2 = dbnvar * (1.0/(n-1)) * torch.ones_like(bndiff2)
cmp("dbndiff2", dbndiff2, bndiff2)
dbndiff = dbnraw * bnvar_inv + dbndiff2 * 2.0 * bndiff
cmp("dbndiff", dbndiff, bndiff)
dbnmeani = (-dbndiff).sum(0)
cmp("dbnmeani", dbnmeani, bnmeani)
dhprebn = dbndiff.clone() + dbnmeani * 1.0/n * torch.ones_like(hprebn)
cmp("dhprebn", dhprebn, hprebn)


# # (m x n)   (m x c)(c x n)
# # hprebn = embcat @ W1 + b1
dembcat = dhprebn @ W1.T
cmp("dembcat", dembcat, embcat)
dW1 = embcat.T @ dhprebn 
cmp("dW1", dW1, W1)
db1 = dhprebn.sum(0)
cmp("db1", db1, b1)

demb = dembcat.view(emb.shape)
cmp("demb", demb, emb)
dC = torch.zeros_like(C)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        i = Xb[k,j]
        dC[i] += demb[k,j] 
cmp("dC", dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff 0.0
dprobs          | exact: True  | approximate: True  | maxdiff 0.0
dcounts_sum_inv | exact: True  | approximate: True  | maxdiff 0.0
dcounts_sum     | exact: True  | approximate: True  | maxdiff 0.0
dcounts         | exact: True  | approximate: True  | maxdiff 0.0
dnorm_logits    | exact: True  | approximate: True  | maxdiff 0.0
dlogit_maxes    | exact: True  | approximate: True  | maxdiff 0.0
dlogits         | exact: True  | approximate: True  | maxdiff 0.0
dh              | exact: True  | approximate: True  | maxdiff 0.0
dW2             | exact: True  | approximate: True  | maxdiff 0.0
db2             | exact: True  | approximate: True  | maxdiff 0.0
dhpreact        | exact: True  | approximate: True  | maxdiff 0.0
dbngain         | exact: True  | approximate: True  | maxdiff 0.0
dbnraw          | exact: True  | approximate: True  | maxdiff 0.0
dbnbias         | exact: True  | approximate: True  | maxdiff 0.0
dbnvar_inv

Note: Bessel's correction to variance computations involves division by $(n-1)$; this is the unbiased variance. 
In the biased variance computation, division is simply by $n$, which almost always will underestimate the variance. 

In [10]:
t = torch.tensor(
    [[1,2,3,4],
    [5,6,7,8],
    [9,10,11,12]]
    )
print(t.shape)
b = t.sum(0, keepdim=True)
b

torch.Size([3, 4])


tensor([[15, 18, 21, 24]])